# Notebook 06 — Pipeline Builder

**Phase 7 demo**: given parsed `Method`, `Dataset`, `Software`, and `Figure` objects, construct an executable pipeline and display the generated Snakefile, Nextflow config, and Jupyter notebook.

Sections:
1. [Setup](#1-setup)
2. [Build mock inputs](#2-build-mock-inputs)
3. [Run PipelineBuilder](#3-run-pipelinebuilder)
4. [Inspect Snakefile](#4-inspect-snakefile)
5. [Inspect Nextflow config](#5-inspect-nextflow-config)
6. [Inspect generated Jupyter notebook](#6-inspect-generated-jupyter-notebook)
7. [Inspect conda environment](#7-inspect-conda-environment)
8. [Save pipeline outputs to disk](#8-save-pipeline-outputs-to-disk)

## 1. Setup

In [ ]:
import json
from pathlib import Path

# researcher_ai imports
from researcher_ai.models.dataset import Dataset, DataSource
from researcher_ai.models.figure import (
    Axis, AxisScale, ColorMapping, ColormapType,
    Figure, PanelLayout, PlotCategory, PlotType, SubFigure,
)
from researcher_ai.models.method import (
    AnalysisStep, Assay, AssayDependency, AssayGraph, Method,
)
from researcher_ai.models.pipeline import PipelineBackend
from researcher_ai.models.software import Command, Environment, LicenseType, Software
from researcher_ai.pipeline import (
    PipelineBuilder, SnakemakeGenerator, NextflowGenerator, JupyterGenerator,
)

print('researcher_ai.pipeline imports OK')

## 2. Build mock inputs

We construct realistic `Method`, `Dataset`, `Software`, and `Figure` objects that
mirror what the upstream parsers (Phases 2–6) would produce for a typical
RNA-seq + ChIP-seq multi-omic paper.

In [ ]:
# ── Method ──────────────────────────────────────────────────────────────────
rnaseq_assay = Assay(
    name="RNA-seq",
    description="Bulk RNA-seq to measure gene expression.",
    data_type="sequencing",
    raw_data_source="GEO: GSE123456",
    steps=[
        AnalysisStep(
            step_number=1,
            description="Quality control with FastQC and adapter trimming with TrimGalore",
            input_data="raw_reads/{sample}_R{1,2}.fastq.gz",
            output_data="trimmed/{sample}_R{1,2}_trimmed.fastq.gz",
            software="TrimGalore",
            software_version="0.6.7",
            parameters={"quality": "20", "length": "20"},
        ),
        AnalysisStep(
            step_number=2,
            description="Alignment to hg38 with STAR",
            input_data="trimmed/{sample}_R{1,2}_trimmed.fastq.gz",
            output_data="aligned/{sample}.Aligned.sortedByCoord.out.bam",
            software="STAR",
            software_version="2.7.10a",
            parameters={"genome": "hg38", "sjdbOverhang": "100", "outSAMtype": "BAM SortedByCoordinate"},
        ),
        AnalysisStep(
            step_number=3,
            description="Read counting with featureCounts",
            input_data="aligned/{sample}.Aligned.sortedByCoord.out.bam",
            output_data="counts/counts_matrix.txt",
            software="featureCounts",
            software_version="2.0.3",
            parameters={"strandedness": "2", "paired": "TRUE"},
        ),
        AnalysisStep(
            step_number=4,
            description="Differential expression with DESeq2",
            input_data="counts/counts_matrix.txt",
            output_data="de_results/deseq2_results.csv",
            software="DESeq2",
            software_version="1.38.0",
            parameters={"padj_threshold": "0.05", "lfc_threshold": "1"},
        ),
    ],
    figures_produced=["Figure 1", "Figure 2"],
)

chipseq_assay = Assay(
    name="ChIP-seq",
    description="H3K27ac ChIP-seq to identify active enhancers.",
    data_type="sequencing",
    raw_data_source="GEO: GSE123457",
    steps=[
        AnalysisStep(
            step_number=1,
            description="Alignment to hg38 with Bowtie2",
            input_data="raw_reads/{sample}_R1.fastq.gz",
            output_data="aligned/{sample}.bam",
            software="Bowtie2",
            software_version="2.5.1",
            parameters={"genome": "hg38"},
        ),
        AnalysisStep(
            step_number=2,
            description="Peak calling with MACS2",
            input_data="aligned/{sample}.bam",
            output_data="peaks/{sample}_peaks.narrowPeak",
            software="MACS2",
            software_version="2.2.7.1",
            parameters={"qvalue": "0.05", "format": "BAM"},
        ),
        AnalysisStep(
            step_number=3,
            description="Annotate peaks with differential expression results",
            input_data="peaks/{sample}_peaks.narrowPeak + de_results/deseq2_results.csv",
            output_data="integrated/annotated_peaks.csv",
            software="ChIPseeker",
            software_version="1.34.0",
            parameters={"TxDb": "TxDb.Hsapiens.UCSC.hg38.knownGene"},
        ),
    ],
    figures_produced=["Figure 3"],
)

method = Method(
    paper_doi="10.1038/s41588-023-99999-9",
    assay_graph=AssayGraph(
        assays=[rnaseq_assay, chipseq_assay],
        dependencies=[
            AssayDependency(
                upstream_assay="RNA-seq",
                downstream_assay="ChIP-seq",
                dependency_type="integration",
                description="ChIP-seq peak annotation uses DESeq2 results for co-localisation analysis",
            )
        ],
    ),
    data_availability="Data available at GEO: GSE123456 (RNA-seq) and GSE123457 (ChIP-seq).",
)

print(f"Method: {method.paper_doi}")
print(f"Assays: {[a.name for a in method.assays]}")
print(f"Total steps: {sum(len(a.steps) for a in method.assays)}")

In [ ]:
# ── Datasets ─────────────────────────────────────────────────────────────────
datasets = [
    Dataset(
        accession="GSE123456",
        source=DataSource.GEO,
        title="RNA-seq of treated vs control cells",
        organism="Homo sapiens",
        experiment_type="RNA-seq",
        total_samples=12,
    ),
    Dataset(
        accession="GSE123457",
        source=DataSource.GEO,
        title="H3K27ac ChIP-seq in treated cells",
        organism="Homo sapiens",
        experiment_type="ChIP-seq",
        total_samples=6,
    ),
]

print(f"Datasets: {[d.accession for d in datasets]}")

In [ ]:
# ── Software ─────────────────────────────────────────────────────────────────
software = [
    Software(
        name="TrimGalore",
        version="0.6.7",
        bioconda_package="trim-galore",
        license_type=LicenseType.OPEN_SOURCE,
        description="Quality control and adapter trimming for FASTQ files.",
        commands=[Command(
            command_template="trim_galore --paired --quality {quality} --length {length} {input}",
            description="Trim adapters and low-quality bases",
        )],
    ),
    Software(
        name="STAR",
        version="2.7.10a",
        bioconda_package="star",
        license_type=LicenseType.OPEN_SOURCE,
        description="Spliced Transcripts Alignment to a Reference.",
        commands=[Command(
            command_template="STAR --runMode alignReads --genomeDir {genome} --readFilesIn {input} --outSAMtype BAM SortedByCoordinate",
            description="Align reads to reference genome",
        )],
    ),
    Software(
        name="featureCounts",
        version="2.0.3",
        bioconda_package="subread",
        license_type=LicenseType.OPEN_SOURCE,
        description="Read summarization for RNA-seq.",
        commands=[Command(
            command_template="featureCounts -p -s {strandedness} -a annotation.gtf -o {output} {input}",
            description="Count reads per gene",
        )],
    ),
    Software(
        name="DESeq2",
        version="1.38.0",
        cran_package=None,
        bioconda_package="bioconductor-deseq2",
        license_type=LicenseType.OPEN_SOURCE,
        language="R",
        description="Differential expression analysis using negative binomial distribution.",
    ),
    Software(
        name="Bowtie2",
        version="2.5.1",
        bioconda_package="bowtie2",
        license_type=LicenseType.OPEN_SOURCE,
        description="Fast and sensitive short-read aligner.",
        commands=[Command(
            command_template="bowtie2 -x {genome} -1 {input[0]} -2 {input[1]} -S {output}.sam",
            description="Align reads to genome",
        )],
    ),
    Software(
        name="MACS2",
        version="2.2.7.1",
        bioconda_package="macs2",
        license_type=LicenseType.OPEN_SOURCE,
        description="Model-based analysis of ChIP-seq data.",
        commands=[Command(
            command_template="macs2 callpeak -t {input} -c {control} -f BAM -g hs -q {qvalue} -n {sample}",
            description="Call peaks from ChIP-seq alignments",
        )],
    ),
]

print(f"Software tools: {[s.name for s in software]}")

In [ ]:
# ── Figures ──────────────────────────────────────────────────────────────────
figures = [
    Figure(
        figure_id="Figure 1",
        title="Differential expression analysis reveals treatment-induced transcriptional changes",
        caption="(a) Volcano plot of differential expression results (DESeq2). Red: padj < 0.05 and |log2FC| > 1. (b) Heatmap of top 50 differentially expressed genes across all samples.",
        purpose="Show the extent and directionality of transcriptional changes upon treatment.",
        layout=PanelLayout(n_rows=1, n_cols=2),
        subfigures=[
            SubFigure(
                label="a",
                description="Volcano plot of DESeq2 results",
                plot_category=PlotCategory.GENOMIC,
                plot_type=PlotType.VOLCANO,
                x_axis=Axis(label="log2 Fold Change", scale=AxisScale.LINEAR),
                y_axis=Axis(label="-log10(adjusted p-value)", scale=AxisScale.LINEAR, is_inverted=True),
                color_mapping=ColorMapping(variable="significance", colormap_type=ColormapType.BINARY),
                data_source="GSE123456",
                assays=["RNA-seq"],
            ),
            SubFigure(
                label="b",
                description="Heatmap of top 50 DEGs",
                plot_category=PlotCategory.MATRIX,
                plot_type=PlotType.HEATMAP,
                x_axis=Axis(label="Sample", scale=AxisScale.CATEGORICAL),
                y_axis=Axis(label="Gene", scale=AxisScale.CATEGORICAL),
                color_mapping=ColorMapping(
                    variable="z-score",
                    colormap_type=ColormapType.DIVERGING,
                    colormap_name="RdBu_r",
                    center_value=0.0,
                ),
                data_source="GSE123456",
                assays=["RNA-seq"],
            ),
        ],
        datasets_used=["GSE123456"],
        methods_used=["RNA-seq"],
    ),
    Figure(
        figure_id="Figure 3",
        title="H3K27ac peaks co-localise with differentially expressed gene promoters",
        caption="(a) Pie chart of genomic feature distribution of H3K27ac peaks. (b) Scatter plot of peak log2FC vs gene log2FC for co-localised features.",
        purpose="Demonstrate that active enhancer marks correspond to transcriptional activation.",
        layout=PanelLayout(n_rows=1, n_cols=2),
        subfigures=[
            SubFigure(
                label="a",
                description="Genomic feature distribution of H3K27ac peaks",
                plot_category=PlotCategory.CATEGORICAL,
                plot_type=PlotType.PIE,
                data_source="GSE123457",
                assays=["ChIP-seq"],
            ),
            SubFigure(
                label="b",
                description="Scatter: peak vs gene log2FC at co-localised loci",
                plot_category=PlotCategory.RELATIONAL,
                plot_type=PlotType.SCATTER,
                x_axis=Axis(label="H3K27ac peak log2FC", scale=AxisScale.LINEAR),
                y_axis=Axis(label="Gene expression log2FC", scale=AxisScale.LINEAR),
                data_source="GSE123456",
                assays=["RNA-seq", "ChIP-seq"],
            ),
        ],
        datasets_used=["GSE123456", "GSE123457"],
        methods_used=["RNA-seq", "ChIP-seq"],
    ),
]

print(f"Figures: {[f.figure_id for f in figures]}")
print(f"Total subfigures: {sum(len(f.subfigures) for f in figures)}")

## 3. Run PipelineBuilder

In [ ]:
builder = PipelineBuilder()

pipeline = builder.build(
    method=method,
    datasets=datasets,
    software=software,
    figures=figures,
    backend=PipelineBackend.SNAKEMAKE,
)

print("Pipeline built successfully!")
print(f"  Name:        {pipeline.config.name}")
print(f"  Backend:     {pipeline.config.backend.value}")
print(f"  nf-core:     {pipeline.config.nf_core_pipeline}")
print(f"  Steps:       {len(pipeline.config.steps)}")
print(f"  Datasets:    {pipeline.config.datasets}")
print(f"  Figures:     {pipeline.config.figure_targets}")
print()
print("Step execution order:")
for step_id in pipeline.config.execution_order():
    step = next(s for s in pipeline.config.steps if s.step_id == step_id)
    deps = f" (depends on: {step.depends_on})" if step.depends_on else ""
    print(f"  {step_id}{deps}")

## 4. Inspect Snakefile

In [ ]:
print(pipeline.snakefile_content)

## 5. Inspect Nextflow config

Build a second pipeline targeting Nextflow to see the nf-core params YAML and samplesheet.

In [ ]:
pipeline_nf = builder.build(
    method=method,
    datasets=datasets,
    software=software,
    figures=figures,
    backend=PipelineBackend.NEXTFLOW,
)

print("=== Nextflow content ===")
print(pipeline_nf.nextflow_content)

## 6. Inspect generated Jupyter notebook

In [ ]:
nb = json.loads(pipeline.jupyter_content)
print(f"Notebook format: {nb['nbformat']}.{nb['nbformat_minor']}")
print(f"Total cells: {len(nb['cells'])}")
print()
for i, cell in enumerate(nb['cells']):
    cell_type = cell['cell_type']
    first_line = cell['source'].splitlines()[0][:80] if cell['source'] else '(empty)'
    print(f"  Cell {i:02d} [{cell_type:8s}]: {first_line}")

In [ ]:
# Preview the setup cell source
print("=== Setup cell ===")
print(nb['cells'][0]['source'])

In [ ]:
# Preview the first plot code cell
code_cells = [c for c in nb['cells'] if c['cell_type'] == 'code']
print(f"Code cells: {len(code_cells)}")
if len(code_cells) > 1:
    print()
    print("=== First plot cell ===")
    print(code_cells[1]['source'])

## 7. Inspect conda environment

In [ ]:
print(pipeline.conda_env_yaml)

## 8. Save pipeline outputs to disk

In [ ]:
output_dir = Path('pipeline_output')
output_dir.mkdir(exist_ok=True)

files_written = []

if pipeline.snakefile_content:
    p = output_dir / 'Snakefile'
    p.write_text(pipeline.snakefile_content)
    files_written.append(str(p))

if pipeline_nf.nextflow_content:
    p = output_dir / 'params.yaml'
    p.write_text(pipeline_nf.nextflow_content)
    files_written.append(str(p))

if pipeline.jupyter_content:
    p = output_dir / 'figure_reproduction.ipynb'
    p.write_text(pipeline.jupyter_content)
    files_written.append(str(p))

if pipeline.conda_env_yaml:
    p = output_dir / 'environment.yaml'
    p.write_text(pipeline.conda_env_yaml)
    files_written.append(str(p))

print('Pipeline outputs saved:')
for f in files_written:
    print(f'  {f}')

In [ ]:
# Confirm JSON schema of the full Pipeline object
import json
pipeline_dict = pipeline.model_dump()
print(f"Pipeline config keys: {list(pipeline_dict['config'].keys())}")
print(f"Number of steps:      {len(pipeline_dict['config']['steps'])}")
print(f"Snakefile length:     {len(pipeline.snakefile_content or '')} chars")
print(f"Nextflow length:      {len(pipeline_nf.nextflow_content or '')} chars")
print(f"Notebook cells:       {len(json.loads(pipeline.jupyter_content)['cells'])}")
print(f"Conda YAML lines:     {len((pipeline.conda_env_yaml or '').splitlines())}")